# Why quantum machine learning?
**Quantum Machine Learning · Week 1 (Data meets qubits) · Day 1**

Lesson page: https://sara-aminpour.com/tutorials/qml/w1d1.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Every quantum machine-learning model in this tutorial follows four steps: **encode** the data into a
quantum state, **process** it with a circuit that has trainable knobs, **measure**, and let a classical
computer **learn** from the results. Here is the smallest complete example.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

Z0 = SparsePauliOp("IZ")                      # Z on qubit 0 (Qiskit reads Pauli strings right to left)

def model(x, knobs):
    qc = QuantumCircuit(2)
    qc.ry(x[0], 0)
    qc.ry(x[1], 1)                            # 1. encode: two features become two rotation angles
    qc.cx(0, 1)                               # 2. process: entangle the qubits...
    qc.ry(knobs[0], 0)
    qc.ry(knobs[1], 1)                        #    ...and turn two trainable knobs
    return Statevector(qc).expectation_value(Z0).real      # 3. measure: <Z> of qubit 0, between -1 and +1

x = np.array([0.4, 2.0])
knobs = np.array([0.3, -0.8])
output = model(x, knobs)
print(f"model output {output:+.3f}  ->  predicted class {int(output < 0)}  (class 1 when <Z> < 0)")

Step 4, **learn**, means choosing the knobs from labeled examples. Training properly comes in week 3.
For now, the crudest possible learner: try many random knob settings and keep the best.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import MinMaxScaler

X, y = make_moons(n_samples=200, noise=0.15, random_state=0)
X = MinMaxScaler((0, np.pi)).fit_transform(X)            # features become angles between 0 and pi

def accuracy(knobs):
    predictions = np.array([int(model(x, knobs) < 0) for x in X])
    return (predictions == y).mean()

rng = np.random.default_rng(0)
tries = rng.uniform(-np.pi, np.pi, size=(300, 2))
scores = [accuracy(k) for k in tries]
best = int(np.argmax(scores))
print(f"random knobs: accuracy between {min(scores):.2f} and {max(scores):.2f}")
print(f"best of 300: {scores[best]:.2f} with knobs {tries[best].round(2)}")

## Your turn: always ask "compared with what?"
Train a plain logistic regression on the same 200 scaled points and compare its accuracy with the
quantum model's best. (Both are measured on the training points here; week 2 does it properly with a
test set.)

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline = None        # TODO: LogisticRegression().fit(X, y), then its .score(X, y)
print("quantum model, best of 300 random knobs:", max(scores), "  logistic regression:", baseline)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
from sklearn.linear_model import LogisticRegression

baseline = LogisticRegression().fit(X, y).score(X, y)
print(f"quantum model, best of 300 random knobs: {max(scores):.3f}")
print(f"logistic regression:                    {baseline:.3f}")
print("A quantum model is not automatically better. Every claim in QML needs a strong classical baseline,")
print("and this tutorial will keep asking for one.")